# Visão Computacional (Artificial)

## O que é?

**Visão Computacional** é a área que busca fazer a máquina **interpretar e compreender** o conteúdo visual de imagens e vídeos — extraindo significado semântico, não apenas transformando pixels. O fluxo é: **Imagem → Interpretação → Decisão/Descrição**.

## Características Principais

- **Entrada:** imagem ou vídeo capturado do mundo real
- **Saída:** informação semântica ("há um rosto aqui", "esse objeto é um carro")
- **Técnicas:** descritores de características (SIFT, HOG), redes convolucionais (CNNs), detecção de objetos (YOLO, R-CNN)
- **Aplicações:** reconhecimento facial, carros autônomos, diagnóstico médico por IA, realidade aumentada

## Aplicações escolhidas

1. **Detecção de Faces** com Haar Cascade (OpenCV) — abordagem clássica baseada em features
2. **Segmentação por Cor** (HSV) — separação de objetos por propriedades cromáticas
3. **Fluxo Óptico** — detectar movimento entre frames consecutivos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
from skimage import data
from skimage.color import rgb2gray
import urllib.request
import os

## 1. Detecção de Faces — Haar Cascade

### Como funciona?

Proposto por Viola e Jones (2001), o classificador Haar Cascade:

1. **Features Haar:** calcula diferenças de intensidade em regiões retangulares adjacentes (como detectar a região mais escura dos olhos em relação à testa)
2. **Integral Image:** permite calcular qualquer soma retangular em O(1)
3. **Cascade de classificadores:** aplica classificadores fracos (stumps) em cascata — regiões sem face são descartadas cedo
4. **Janela deslizante:** varre a imagem em múltiplas escalas (pirâmide de imagem)

In [ ]:
# Carrega o classificador pré-treinado do OpenCV
cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
eye_cascade_path = cv2.data.haarcascades + 'haarcascade_eye.xml'

face_cascade = cv2.CascadeClassifier(cascade_path)
eye_cascade  = cv2.CascadeClassifier(eye_cascade_path)

# Imagem de astronauta do scikit-image (tem rosto frontal)
img_rgb = data.astronaut()
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

# Detecção de faces
faces = face_cascade.detectMultiScale(
    img_gray,
    scaleFactor=1.05,  # quanto a imagem é reduzida a cada escala
    minNeighbors=3,    # mínimo de vizinhos para confirmar detecção
    minSize=(30, 30)
)

print(f"Faces detectadas: {len(faces)}")
if len(faces) > 0:
    print(f"Bounding boxes (x, y, w, h): {faces}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original
axes[0].imshow(img_rgb)
axes[0].set_title('Imagem Original', fontsize=11)
axes[0].axis('off')

# Com detecções
img_deteccao = img_rgb.copy()
ax = axes[1]
ax.imshow(img_deteccao)
for (x, y, w, h) in faces:
    rect = patches.Rectangle((x, y), w, h, linewidth=2,
                               edgecolor='lime', facecolor='none')
    ax.add_patch(rect)
    ax.text(x, y-5, 'Rosto', color='lime', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))

    # Busca olhos dentro da região do rosto
    roi_gray = img_gray[y:y+h, x:x+w]
    eyes = eye_cascade.detectMultiScale(roi_gray, scaleFactor=1.1, minNeighbors=5)
    for (ex, ey, ew, eh) in eyes:
        rect_eye = patches.Rectangle((x+ex, y+ey), ew, eh, linewidth=1.5,
                                      edgecolor='cyan', facecolor='none')
        ax.add_patch(rect_eye)

ax.set_title(f'Detecção: {len(faces)} rosto(s) (verde) + olhos (ciano)', fontsize=11)
ax.axis('off')

# Pirâmide de escala (visualização)
pyramid = [img_gray]
tmp = img_gray.copy()
for _ in range(3):
    tmp = cv2.pyrDown(tmp)
    pyramid.append(tmp)

# Monta pirâmide lado a lado
widths = [p.shape[1] for p in pyramid]
total_w = sum(widths) + 10 * len(pyramid)
max_h = pyramid[0].shape[0]
canvas = np.ones((max_h, total_w), dtype=np.uint8) * 200
x_off = 0
for p in pyramid:
    canvas[:p.shape[0], x_off:x_off+p.shape[1]] = p
    x_off += p.shape[1] + 10

axes[2].imshow(canvas, cmap='gray')
axes[2].set_title('Pirâmide de Escala (busca multi-escala)', fontsize=11)
axes[2].axis('off')

plt.suptitle('Detecção de Faces — Haar Cascade (Viola-Jones, 2001)', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Features Haar — Visualização

As features Haar exploram o fato de que regiões como os olhos tendem a ser mais escuras que a testa e as bochechas.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))

haar_names = [
    'Borda\nhorizontal',
    'Borda\nvertical',
    'Borda\ndiagonal',
    'Linha\nhorizontal',
    'Linha\nvertical',
]

haar_kernels = [
    np.array([[1, 1], [-1, -1]], dtype=float),
    np.array([[1, -1], [1, -1]], dtype=float),
    np.array([[1, -1], [-1, 1]], dtype=float),
    np.array([[1, 1, 1], [-1, -1, -1]], dtype=float),
    np.array([[1, -1, 1], [1, -1, 1]], dtype=float),
]

for ax, k, name in zip(axes, haar_kernels, haar_names):
    im = ax.imshow(k, cmap='RdBu', interpolation='nearest', vmin=-1, vmax=1)
    ax.set_title(name, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    for (i, j), val in np.ndenumerate(k):
        ax.text(j, i, '+' if val > 0 else '-', ha='center', va='center',
                fontsize=14, fontweight='bold',
                color='white' if abs(val) > 0.5 else 'black')

plt.suptitle('Tipos de Features Haar (branco=+1, azul=-1)', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Segmentação por Cor no Espaço HSV

O espaço **HSV** (Matiz, Saturação, Valor) separa a informação de cor (H) da luminosidade (V), tornando a segmentação por cor muito mais robusta a variações de iluminação do que o RGB.

- **H (Hue):** ângulo no círculo cromático [0°, 360°]
- **S (Saturation):** pureza da cor [0, 1]
- **V (Value):** brilho [0, 1]

In [ ]:
# Criamos uma imagem sintética com objetos coloridos
img_sintetica = np.zeros((300, 400, 3), dtype=np.uint8)

# Círculo vermelho
cv2.circle(img_sintetica, (80, 150), 60, (200, 50, 50), -1)
# Retângulo verde
cv2.rectangle(img_sintetica, (160, 80), (260, 220), (50, 180, 50), -1)
# Triângulo azul
pts = np.array([[320, 80], [280, 220], [370, 220]], np.int32)
cv2.fillPoly(img_sintetica, [pts], (50, 50, 200))
# Fundo cinza
mask_objetos = (img_sintetica.sum(axis=2) == 0)
img_sintetica[mask_objetos] = [128, 128, 128]

# Converte para HSV
img_hsv = cv2.cvtColor(img_sintetica, cv2.COLOR_BGR2HSV)
img_rgb_sintetica = cv2.cvtColor(img_sintetica, cv2.COLOR_BGR2RGB)

# Máscaras de segmentação por cor
mask_vermelho1 = cv2.inRange(img_hsv, np.array([0,   100, 100]), np.array([10,  255, 255]))
mask_vermelho2 = cv2.inRange(img_hsv, np.array([170, 100, 100]), np.array([180, 255, 255]))
mask_vermelho = cv2.bitwise_or(mask_vermelho1, mask_vermelho2)

mask_verde  = cv2.inRange(img_hsv, np.array([35, 80, 80]),  np.array([85,  255, 255]))
mask_azul   = cv2.inRange(img_hsv, np.array([100, 80, 80]), np.array([130, 255, 255]))

# Aplica morfologia para suavizar máscaras
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
for mask in [mask_vermelho, mask_verde, mask_azul]:
    cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, dst=mask)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

axes[0,0].imshow(img_rgb_sintetica)
axes[0,0].set_title('Imagem Original (RGB)', fontsize=11)

axes[0,1].imshow(img_hsv[:,:,0], cmap='hsv')
axes[0,1].set_title('Canal H (Matiz)', fontsize=11)

axes[0,2].imshow(img_hsv[:,:,1], cmap='gray')
axes[0,2].set_title('Canal S (Saturação)', fontsize=11)

axes[0,3].imshow(img_hsv[:,:,2], cmap='gray')
axes[0,3].set_title('Canal V (Valor/Brilho)', fontsize=11)

axes[1,0].imshow(mask_vermelho, cmap='Reds')
axes[1,0].set_title('Máscara: Vermelho', fontsize=11)

axes[1,1].imshow(mask_verde, cmap='Greens')
axes[1,1].set_title('Máscara: Verde', fontsize=11)

axes[1,2].imshow(mask_azul, cmap='Blues')
axes[1,2].set_title('Máscara: Azul', fontsize=11)

# Resultado final: cada objeto com cor original sobre fundo cinza
resultado = np.ones_like(img_rgb_sintetica) * 200
for mask, canal in [(mask_vermelho, None), (mask_verde, None), (mask_azul, None)]:
    m3 = np.stack([mask]*3, axis=2) > 0
    resultado[m3] = img_rgb_sintetica[m3]

axes[1,3].imshow(resultado)
axes[1,3].set_title('Objetos Segmentados', fontsize=11)

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Segmentação por Cor no Espaço HSV', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Detector de Keypoints — ORB

**ORB** (Oriented FAST + Rotated BRIEF) é um detector de pontos de interesse eficiente, livre de patentes, usado em correspondência de imagens, SLAM e realidade aumentada.

In [ ]:
img_gray_astro = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

orb = cv2.ORB_create(nfeatures=200)
keypoints, descriptors = orb.detectAndCompute(img_gray_astro, None)

# Imagem com keypoints desenhados
img_kp = cv2.drawKeypoints(
    img_rgb, keypoints, None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

# Mapa de densidade de keypoints
kp_map = np.zeros(img_gray_astro.shape, dtype=np.float32)
for kp in keypoints:
    x, y = int(kp.pt[0]), int(kp.pt[1])
    r = max(1, int(kp.size / 4))
    cv2.circle(kp_map, (x, y), r, 1.0, -1)

kp_map = cv2.GaussianBlur(kp_map, (31, 31), 10)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_rgb)
axes[0].set_title('Original', fontsize=11)
axes[0].axis('off')

axes[1].imshow(img_kp)
axes[1].set_title(f'Keypoints ORB ({len(keypoints)} pontos)\n(círculo = escala, linha = orientação)', fontsize=11)
axes[1].axis('off')

axes[2].imshow(img_rgb)
axes[2].imshow(kp_map, cmap='hot', alpha=0.6)
axes[2].set_title('Mapa de Densidade de Keypoints\n(regiões com mais textura/contraste)', fontsize=11)
axes[2].axis('off')

plt.suptitle('Detecção de Pontos de Interesse — ORB', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Descritores: shape={descriptors.shape}, dtype={descriptors.dtype}")
print(f"Cada keypoint tem um descritor binário de {descriptors.shape[1]*8} bits (BRIEF)")

## Conclusão

**Visão Computacional** difere das outras áreas pelo objetivo de **compreender** o conteúdo da imagem:

| Aspecto | Visão Computacional |
|---------|--------------------|
| **Entrada** | Imagem real (câmera, sensor) |
| **Saída** | Interpretação semântica ("rosto", "objeto vermelho", "ponto de interesse") |
| **Operação** | Extração de features + classificação/localização |
| **Exemplos** | Face ID, detecção de pedestres, leitura de placas |

Diferentemente do **Processamento de Imagens** (que melhora pixels), a Visão Computacional quer **responder perguntas** sobre a imagem — "o que há aqui?", "onde está?", "o que mudou?".